In [1]:
!pip install pytorchvideo torchvision av

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 132.7/132.7 kB 1.2 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.2/50.2 kB 1.8 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.2/42.2 kB 1.9 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.4/35.4 MB 43.1 MB/s eta 0:00:00
  Created wheel for pytorchvideo: filename=pytorchvideo-0.1.5-py3-none-any.whl size=188686 sha256=d462936dfd0e9ab2cfe27eab5d48eb2c63315cd6aad9b438f7bcbd49d20e6239
  Stored in directory: /root/.cache/pip/wheels/b3/49/dc/aab2dce83e38b59849db13a4f4ddd220e568e24b58332fb0f9
  Created wheel for fvcore: filename=fvcore-0.1.5.post20221221-py3-none-any.whl size=61397 sha256=e6ae44e9f6d29319091a6efa1f352434de0c43f2b6350e268d888a8fcaae6fe2
  Stored in directory: /root/.cache/pip/wheels/ed/9f/a5/e4f5b27454ccd4596bd8b62432c7d6b1ca9fa22aef9d70a16a
  Created wheel f

In [2]:
# --- 0. THE PYTORCHVIDEO BUG FIX (Monkey Patch) ---
import sys
import torchvision.transforms.functional as F_vision
# Trick pytorchvideo into thinking the old deprecated module still exists
sys.modules['torchvision.transforms.functional_tensor'] = F_vision

# --- 1. IMPORTS ---
import os
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from google.colab import drive

import pytorchvideo.data
from pytorchvideo.transforms import ApplyTransformToKey, UniformTemporalSubsample
from torchvision.transforms import Compose, Lambda, Resize
from torchvision.transforms._transforms_video import NormalizeVideo
from pytorchvideo.data.encoded_video import EncodedVideo

# --- 2. CONFIGURATION ---
drive.mount('/content/drive', force_remount=True)

DATA_PATH = "/content/drive/MyDrive/video_data/train"
CLASSES = ['walking', 'running']
BATCH_SIZE = 2
NUM_FRAMES = 8  # Slow-R50 traditionally expects 8 frames
EPOCHS = 3
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# --- 3. ROBUST DATASET DEFINITION ---
labeled_video_paths = []
for i, cls_name in enumerate(CLASSES):
    cls_path = os.path.join(DATA_PATH, cls_name)
    if not os.path.isdir(cls_path): continue
    for f in os.listdir(cls_path):
        if f.lower().endswith(('.mp4', '.avi', '.mov')):
            labeled_video_paths.append((os.path.join(cls_path, f), {'label': i}))

# Slow-R50 Transform Pipeline
video_transform = Compose([
    ApplyTransformToKey(
        key="video",
        transform=Compose([
            UniformTemporalSubsample(NUM_FRAMES),
            Lambda(lambda x: x / 255.0), # Normalize to 0-1
            # Slow-R50 uses these specific normalization values
            NormalizeVideo(mean=[0.45, 0.45, 0.45], std=[0.225, 0.225, 0.225]),
            # Slow-R50 default crop size is 256x256
            Resize((256, 256))
        ])
    )
])

# Safely extract 2-second clips
clip_sampler = pytorchvideo.data.make_clip_sampler("uniform", 2.0)

train_dataset = pytorchvideo.data.LabeledVideoDataset(
    labeled_video_paths=labeled_video_paths,
    clip_sampler=clip_sampler,
    transform=video_transform,
    decode_audio=False
)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE)

# --- 4. MODEL INITIALIZATION ---
print("Downloading Slow-R50 model from PyTorch Hub...")
model = torch.hub.load('facebookresearch/pytorchvideo', 'slow_r50', pretrained=True)

# Replace the classification head for Slow-R50. In Slow-R50, it's the 5th block.
in_features = model.blocks[5].proj.in_features
model.blocks[5].proj = nn.Linear(in_features, len(CLASSES))

model = model.to(DEVICE)

# --- 5. TRAINING LOOP ---
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4)
criterion = nn.CrossEntropyLoss()

print("\n--- Starting Slow-R50 Training ---")
model.train()

for epoch in range(EPOCHS):
    running_loss = 0.0
    batches = 0

    for batch_dict in train_loader:
        # Slow-R50 expects shape: (Batch, Channels, Time, Height, Width)
        videos = batch_dict['video'].to(DEVICE)
        labels = batch_dict['label'].to(DEVICE)

        outputs = model(videos)
        loss = criterion(outputs, labels)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        running_loss += loss.item()
        batches += 1

    avg_loss = running_loss / batches if batches > 0 else 0
    print(f"Epoch [{epoch+1}/{EPOCHS}] Average Loss: {avg_loss:.4f}")

# --- 5.1 SAVING THE TRAINED MODEL ---
save_path = "/content/drive/MyDrive/video_slow_r50_TrainedModel.pth"
torch.save(model.state_dict(), save_path)
print(f"Yay! Model safely saved to: {save_path}")

# --- 6. TARGET FILE / INFERENCE WITH THRESHOLD ---
NEW_VIDEO_PATH = "/content/drive/MyDrive/video_data/test/v_PlayingCello_g04_c02.avi"

def run_guaranteed_inference(video_path, threshold=0.95):
    """
    Runs inference and intercepts low-confidence predictions
    to label them as 'MISCELLANEOUS'.
    """
    model.eval()
    try:
        video = EncodedVideo.from_path(video_path)
        video_data = video.get_clip(start_sec=0.0, end_sec=2.0)

        # Apply the exact same transform
        video_data = video_transform(video_data)

        # Add batch dimension and send to GPU
        input_tensor = video_data["video"].unsqueeze(0).to(DEVICE)

        with torch.no_grad():
            output = model(input_tensor)
            probs = F.softmax(output, dim=1)
            conf, pred_idx = torch.max(probs, 1)

        # Extract the values
        confidence_score = conf.item()
        predicted_class = CLASSES[pred_idx.item()].upper()

        print("-" * 30)
        print(f"FILE: {os.path.basename(video_path)}")

        # --- THE MISCELLANEOUS INTERCEPT ---
        if confidence_score < threshold:
            print(f"PREDICTED: MISCELLANEOUS (Confidence below {threshold*100}%)")
            print(f"ORIGINAL GUESS: {predicted_class} at {confidence_score*100:.2f}%")
        else:
            print(f"PREDICTED: {predicted_class}")
            print(f"CONFIDENCE: {confidence_score*100:.2f}%")

        print("-" * 30)

    except Exception as e:
        print(f"Inference failed: {e}")

# Run the inference on your cello video.
# You can adjust the threshold here (e.g., 0.85, 0.90, 0.95, 0.99)
run_guaranteed_inference(NEW_VIDEO_PATH, threshold=0.70)

/usr/local/lib/python3.12/dist-packages/torchvision/transforms/_functional_video.py:6: UserWarning: The 'torchvision.transforms._functional_video' module is deprecated since 0.12 and will be removed in the future. Please use the 'torchvision.transforms.functional' module instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/transforms/_transforms_video.py:22: UserWarning: The 'torchvision.transforms._transforms_video' module is deprecated since 0.12 and will be removed in the future. Please use the 'torchvision.transforms' module instead.
  warnings.warn(


Mounted at /content/drive
Downloading: "https://github.com/facebookresearch/pytorchvideo/zipball/main" to /root/.cache/torch/hub/main.zip
Downloading: "https://dl.fbaipublicfiles.com/pytorchvideo/model_zoo/kinetics/SLOW_8x8_R50.pyth" to /root/.cache/torch/hub/checkpoints/SLOW_8x8_R50.pyth


100%|██████████| 248M/248M [00:08<00:00, 30.9MB/s]



--- Starting Slow-R50 Training ---
Epoch [1/3] Average Loss: 0.4507
Epoch [2/3] Average Loss: 0.0655
Epoch [3/3] Average Loss: 0.0165
Yay! Model safely saved to: /content/drive/MyDrive/video_slow_r50_TrainedModel.pth
------------------------------
FILE: v_PlayingCello_g04_c02.avi
PREDICTED: RUNNING
CONFIDENCE: 98.40%
------------------------------
